# Text-to-Image (TTI) — image generation benchmark on Apple Silicon

Generates one 1024 × 1024 image per prompt with each model in **float16** on MLX and records, per prompt: resolution,
steps, guidance, prompt length, time to first step, time per step, inference rate and inference time, plus system
metrics (CPU / GPU / NPU utilisation, power, temperature) sampled by `macmon`. A small vision model then captions each
image, so what was asked for can be compared with what was drawn.

| Model | MLX library |
|---|---|
| stable-diffusion-xl-base-1.0 | Apple's MLX Stable Diffusion code from `mlx-examples`, in `tti_backends/mlx_stable_diffusion/` |
| FLUX.2-klein-4B | [`mflux`](https://github.com/filipstrand/mflux) |

Each model is kept as a local float16 copy at `models/<model>-fp16/` (SDXL: the repo's own fp16 files; FLUX.2-klein:
loaded in float16 by mflux and saved). Two parts can't run in float16 and keep a wider type: SDXL's VAE runs in float32,
and FLUX.2-klein's latents during denoising stay bfloat16 (in float16 both overflow into black images).

**Outputs**
- `outputs/tti/<family>/<params>/<prompt set>/`: one `.png` per prompt and `<model>_simple_metrics.json`
- `metrics_analysis/tti/<family>/<params>/<prompt set>/`: `Model_Level_Metrics/<model>_prompt_metrics.csv`
  and `System_Level_Metrics/<prompt id>_system_metrics.csv`

## 1. Settings and model registry

In [ ]:
import gc
import json
import os
import sys
import time
from pathlib import Path

import mlx.core as mx
import numpy as np
import pandas as pd
from PIL import Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'prompts').is_dir())   # project root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault('HF_HUB_DISABLE_IMPLICIT_TOKEN', '1')   # every model here is public
mx.set_cache_limit(4 * 1024**3)   # cap MLX's buffer cache: at 1024 x 1024 it otherwise grows until macOS swaps

# family                 size      params  model                           source on Hugging Face                     params (B)  library         steps  guidance
REGISTRY = [
    ('Stable-Diffusion-XL', 'MEDIUM', '3.5B', 'stable-diffusion-xl-base-1.0', 'stabilityai/stable-diffusion-xl-base-1.0', 3.47, 'mlx-examples', 30,    7.0),
    ('FLUX.2-klein',        'SMALL',  '4B',   'FLUX.2-klein-4B',              'black-forest-labs/FLUX.2-klein-4B',        4.0,  'mflux',        4,     1.0),
]
MODELS = pd.DataFrame(REGISTRY, columns=['family', 'size', 'params', 'model', 'source', 'params_b',
                                         'library', 'steps', 'guidance']).set_index('model')
MODELS['path'] = 'models/' + MODELS.index + '-fp16'   # local float16 copy of each model

RUN = os.environ.get('TTI_MODELS', ','.join(MODELS.index)).split(',')   # models to run, in order
PROMPT_SET = os.environ.get('TTI_PROMPTS', 'tti_prompts_10')            # prompts/tti_tts_ttt_prompts/<set>.csv (id, prompt)
WIDTH = HEIGHT = 1024                                # native resolution of both models
SEED = 0                                             # same seed for every prompt and model, so runs are repeatable
CAPTION_MODEL = 'models/Qwen2-VL-2B-Instruct-fp16'   # float16 vision model from the ITT notebook, captions each image

print(f'Models: {RUN}  |  {WIDTH} x {HEIGHT}, seed {SEED}')
MODELS[['family', 'size', 'params', 'library', 'source', 'path', 'steps', 'guidance']]

## 2. Prepare and load the models

The first time, each model's float16 copy is created in `models/<model>-fp16/` (one-time step); after that it loads
from there.

In [ ]:
from huggingface_hub import snapshot_download
from mflux.models.common.config import ModelConfig
from mflux.models.flux2.variants.txt2img.flux2_klein import Flux2Klein
from tti_backends.mlx_stable_diffusion import StableDiffusionXL
from tti_backends.mlx_stable_diffusion import model_io

def prepare(name):
    m = MODELS.loc[name]
    if Path(m.path).exists():
        return
    print(f'Preparing {m.path} from {m.source} (float16), one-time step...')
    if m.library == 'mlx-examples':   # copy just the files the loader uses (the repo's fp16 weights)
        snapshot_download(m.source, token=False, local_dir=m.path,
                          allow_patterns=list(model_io._MODELS[m.source].values()))
    else:                             # load in float16 and save the float16 weights
        ModelConfig.precision = mx.float16
        Flux2Klein(model_config=ModelConfig.flux2_klein_4b()).save_model(m.path)

def load_model(name):
    m = MODELS.loc[name]
    prepare(name)
    t0 = time.perf_counter()
    if m.library == 'mlx-examples':
        model_io.LOCAL_PATHS[m.source] = m.path   # read the files from models/ instead of the Hugging Face cache
        model = StableDiffusionXL(m.source, float16=True)
        model.autoencoder = model_io.load_autoencoder(m.source, float16=False)   # SDXL's VAE overflows in float16
        model.ensure_models_are_loaded()
    else:
        ModelConfig.precision = mx.float16    # mflux loads the weights in this dtype (its default is bfloat16)
        model = Flux2Klein(model_path=m.path, model_config=ModelConfig.flux2_klein_4b())
        mx.eval(model.parameters())           # mflux loads lazily; load now so load time and memory are real
        ModelConfig.precision = mx.bfloat16   # ...but its latents must stay bfloat16: in float16 they overflow (black images)
    return model, time.perf_counter() - t0

## 3. Prompts

In [ ]:
PROMPTS = pd.read_csv(Path('prompts/tti_tts_ttt_prompts') / f'{PROMPT_SET}.csv', dtype={'id': str})
PROMPTS['id'] = PROMPTS.id.str.zfill(2)
print(f'{len(PROMPTS)} prompts from {PROMPT_SET}.csv')
PROMPTS

## 4. Generate the images

Each model is loaded once and warmed up with one untimed image (so GPU kernels are compiled before anything is timed).
Each prompt is wrapped in `metrics_monitor_start()` / `metrics_monitor_stop()` (from `macmon_metrics/`), which saves
that image's system metrics every 100 ms. After generation (outside the timed windows) the vision model writes a
one-sentence caption of every image.

`time_to_first_step (ms)` is text encoding plus the first denoising step (SDXL only: mflux doesn't expose its steps);
`time_per_step (ms)` is inference time ÷ steps.

In [ ]:
import macmon_metrics.metrics_monitor as mm
from macmon_metrics.metrics_monitor import metrics_monitor_start, metrics_monitor_stop, setup_output_directory
from mlx_vlm import load as load_vlm, generate as vlm_generate
from mlx_vlm.prompt_utils import apply_chat_template

if mm._tracker is None:   # start macmon now, so every image's capture has samples from its first moment
    mm._tracker = mm.MacMonTracker(binary_path=mm._resolve_macmon_binary())
    mm._tracker.start()
    time.sleep(1.5)

def generate(model, m, prompt):
    # One image; returns it with its timings (first step and VAE decode for SDXL)
    mx.reset_peak_memory()
    t0 = time.perf_counter()
    first = decode = None
    if m.library == 'mlx-examples':
        for x_t in model.generate_latents(prompt, n_images=1, num_steps=int(m.steps), cfg_weight=m.guidance,
                                          latent_size=(HEIGHT // 8, WIDTH // 8), seed=SEED):
            mx.eval(x_t)
            first = first or time.perf_counter() - t0
        t_denoised = time.perf_counter()
        x = model.decode(x_t.astype(mx.float32))
        mx.eval(x)
        decode = time.perf_counter() - t_denoised
        image = Image.fromarray((np.array(x[0]) * 255).astype(np.uint8))
    else:
        image = model.generate_image(seed=SEED, prompt=prompt, num_inference_steps=int(m.steps),
                                     height=HEIGHT, width=WIDTH, guidance=m.guidance).image
    total = time.perf_counter() - t0
    mx.clear_cache()   # release this image's buffers before the next one
    return image, total, first, decode, mx.get_peak_memory() / 1e9

def caption(paths):
    # One-sentence description of each image by the vision model, then unload it
    vlm, proc = load_vlm(CAPTION_MODEL)
    question = apply_chat_template(proc, vlm.config, 'Describe this image in one sentence.', num_images=1)
    out = [vlm_generate(vlm, proc, question, image=[p], max_tokens=80, temperature=0.0, verbose=False).text.strip()
           for p in paths]
    del vlm, proc
    gc.collect()
    mx.clear_cache()
    return out

def run_model(name):
    # Generate one image per prompt; save images, per-prompt metrics and a summary
    m = MODELS.loc[name]
    sub = Path(m.family) / m.params / PROMPT_SET
    out_dir = Path('outputs/tti') / sub
    out_dir.mkdir(parents=True, exist_ok=True)
    sys_dir, model_dir = setup_output_directory(str(Path('metrics_analysis/tti') / sub))
    model, load_s = load_model(name)
    generate(model, m, PROMPTS.prompt[0])   # warm-up
    rows, decodes, peak = [], [], 0.0
    for p in PROMPTS.itertuples():
        metrics_monitor_start(idx=p.id, metrics_folder=sys_dir)
        image, total, first, decode, peak_gb = generate(model, m, p.prompt)
        metrics_monitor_stop()
        path = out_dir / f'{name}_image_{p.id}.png'
        image.save(path)
        decodes.append(decode)
        peak = max(peak, peak_gb)
        rows.append({
            'prompt_id': p.id, 'model': name, 'family': m.family, 'size': m['size'],
            'resolution (px)': f'{WIDTH}x{HEIGHT}', 'steps': int(m.steps), 'guidance': m.guidance, 'seed': SEED,
            'input_words': len(p.prompt.split()), 'input_characters': len(p.prompt),
            'time_to_first_step (ms)': round(1000 * first, 1) if first else None,
            'time_per_step (ms)': round(1000 * total / m.steps, 1),
            'inference_rate (images/min)': round(60 / total, 2),
            'inference_time (s)': round(total, 3),
            'image_file': str(path), 'prompt': p.prompt,
        })
        print(f'  [{p.id}] {total:.1f} s ({1000 * total / m.steps:.0f} ms/step)  {p.prompt[:60]}')
    del model
    gc.collect()
    mx.clear_cache()

    df = pd.DataFrame(rows)
    df['caption'] = caption(df.image_file)
    df.to_csv(Path(model_dir) / f'{name}_prompt_metrics.csv', index=False)
    hw = pd.concat([pd.read_csv(Path(sys_dir) / f'{i}_system_metrics.csv') for i in df.prompt_id])
    summary = {
        'model': name, 'family': m.family, 'size': m['size'], 'parameters': m.params, 'precision': 'float16',
        'library': m.library, 'path': m.path, 'source': m.source, 'prompt_set': PROMPT_SET, 'images': len(df),
        'resolution (px)': f'{WIDTH}x{HEIGHT}', 'steps': int(m.steps), 'guidance': m.guidance, 'seed': SEED,
        'model_load (s)': round(load_s, 1),
        'time_to_first_step (ms, avg)': round(df['time_to_first_step (ms)'].mean(), 1) if m.library == 'mlx-examples' else None,
        'time_per_step (ms, avg)': round(df['time_per_step (ms)'].mean(), 1),
        'vae_decode (s, avg)': round(np.mean(decodes), 2) if m.library == 'mlx-examples' else None,
        'inference_time (s, avg)': round(df['inference_time (s)'].mean(), 2),
        'inference_time (s, total)': round(df['inference_time (s)'].sum(), 2),
        'inference_rate (images/min, avg)': round(df['inference_rate (images/min)'].mean(), 2),
        'peak_memory (GB)': round(peak, 2),
        'gpu_utilization (%, avg)': round(hw['GPU Utilization (%)'].mean(), 1),
        'gpu_power (W, avg)': round(hw['gpuPower'].mean(), 1),
        'gpu_temperature (C, max)': round(hw['GPU_Temperature(C)'].max(), 1),
        'cpu_utilization (%, avg)': round(hw['CPU_Util(%)'].mean(), 1),
        'cpu_power (W, avg)': round(hw['cpuPower'].mean(), 1),
        'npu_power (W, avg)': round(hw['npuPower'].mean(), 2),
        'system_energy (J)': round(hw['total_system_power'].sum() * 0.1, 1),   # 100 ms samples
    }
    (out_dir / f'{name}_simple_metrics.json').write_text(json.dumps(summary, indent=2))
    return summary, df

results = {}
for name in RUN:
    print(name)
    results[name], per_prompt = run_model(name)
    display(per_prompt[['prompt_id', 'inference_time (s)', 'time_per_step (ms)', 'prompt', 'caption']])
pd.DataFrame(results)

## 5. All models run so far

Combines every float16 model run on `PROMPT_SET` into `outputs/tti/all_models_<prompt set>.csv` and their per-prompt
metrics into `metrics_analysis/tti/all_models_prompt_metrics_<prompt set>.csv`.

In [ ]:
summaries = [json.loads(f.read_text()) for f in Path('outputs/tti').glob(f'*/*/{PROMPT_SET}/*_simple_metrics.json')]
summaries = sorted((s for s in summaries if s.get('precision') == 'float16'), key=lambda s: list(MODELS.index).index(s['model']))
all_models = pd.DataFrame(summaries)
all_models.to_csv(Path('outputs/tti') / f'all_models_{PROMPT_SET}.csv', index=False)
all_prompts = pd.concat([
    pd.read_csv(Path('metrics_analysis/tti') / s['family'] / s['parameters'] / PROMPT_SET / 'Model_Level_Metrics' /
                f"{s['model']}_prompt_metrics.csv", dtype={'prompt_id': str})
    for s in summaries])
all_prompts.to_csv(Path('metrics_analysis/tti') / f'all_models_prompt_metrics_{PROMPT_SET}.csv', index=False)
print(f'{len(all_models)} models, {len(all_prompts)} prompt rows')
all_models